# 04b - SARIMA Forecasting

**Objective**: Forecast using SARIMA (Seasonal ARIMA)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima
import warnings
warnings.filterwarnings('ignore')
print('✓ Libraries imported')

In [ ]:
# Load datatry:    df = pd.read_csv('../data/clean/maize_features.csv')except FileNotFoundError:    df = pd.read_csv('data/clean/maize_features.csv')df = df[['date', 'price']].set_index('date')print(f'Dataset: {df.shape}')

In [ ]:
# Train-test split
train = df.iloc[:-12]
test = df.iloc[-12:]
print(f'Train: {len(train)}, Test: {len(test)}')

In [ ]:
# Auto ARIMA to find best parameters
stepwise_model = auto_arima(train, start_p=1, start_q=1, max_p=3, max_q=3, m=12, start_P=0, seasonal=True, d=None, D=1, trace=True, error_action='ignore', suppress_warnings=True, stepwise=True)
print(f'\nBest model: {stepwise_model.order} x {stepwise_model.seasonal_order}')

In [ ]:
# Train SARIMA with best parameters
model = SARIMAX(train, order=stepwise_model.order, seasonal_order=stepwise_model.seasonal_order)
model_fit = model.fit(disp=False)
print('✓ SARIMA model trained')
print(model_fit.summary())

In [ ]:
# Forecast
forecast = model_fit.forecast(steps=12)
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test['price'] - forecast) / test['price'])) * 100
print(f'MAE: {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'MAPE: {mape:.2f}%')

In [ ]:
# Save model
import joblib
joblib.dump(model_fit, '../models/trained/sarima_maize_model.pkl')
forecast_df = pd.DataFrame({'date': test.index, 'forecast': forecast.values})
forecast_df.to_csv('../models/trained/sarima_forecast.csv', index=False)
print('✓ Model saved')